# Analytická část – průzkum dat

Průzkumová analýza vyčištěné verzované historie stavebních objektů z RÚIAN pro Brno a okolí.
Notebook vytváří tabulky (`tabulky/`) a obrázky (`obrazky/`) pro kapitolu bakalářské práce.
Osnova kapitoly je v [OSNOVA.md](OSNOVA.md).

Tento díl: **1. Charakteristika dat** a **2. Struktura historie**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import shapely
from matplotlib.colors import LinearSegmentedColormap

ROOT = Path.cwd()
if ROOT.name != "analyticka_cast":
    ROOT = ROOT / "analyza" / "analyticka_cast"
REPO = ROOT.parents[1]

DATA = REPO / "navrh" / "clean_output" / "brno_area_history_clean.parquet"
CODELISTS = REPO / "analyza" / "ciselniky"
FIG_DIR = ROOT / "obrazky"
TABLE_DIR = ROOT / "tabulky"
FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR.mkdir(exist_ok=True)

BLUE, ORANGE, AQUA, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#a9a8a3"
INK_SOFT, GRID = "#52514e", "#e4e3df"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 220, "savefig.bbox": "tight",
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_SOFT, "axes.titlesize": 10,
    "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.axisbelow": True, "xtick.color": INK_SOFT, "ytick.color": INK_SOFT,
    "legend.frameon": False, "figure.figsize": (6.3, 3.2),
})


def thousands(x, _pos=None):
    return f"{x:,.0f}".replace(",", "\u00a0")


def pct(x):
    return f"{x:.0%}".replace("%", "\u00a0%")


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png")
    fig.savefig(FIG_DIR / f"{name}.pdf")

## Načtení dat

Vyčištěný výstup pipeline: jeden řádek = jedna verze stavebního objektu s platností
`valid_from` – `valid_to` (`valid_to` prázdné = aktuální verze). Geometrie zatím nepotřebujeme,
načítáme jen atributy.

In [ ]:
cols = [
    "code", "building_type_code", "usage_type_code", "completion_date", "construction_type_code",
    "unit_count", "floor_count", "sewage_connection_code", "gas_connection_code",
    "water_connection_code", "elevator_code", "built_up_area", "heating_type_code",
    "valid_from", "valid_to", "end_reason", "municipality_code", "municipality_name",
    "snapshot_date", "reference_point_wgs84",
]
df = pd.read_parquet(DATA, columns=cols)
for c in ["valid_from", "valid_to", "completion_date", "snapshot_date"]:
    df[c] = pd.to_datetime(df[c])
# Validity dates after a missing monthly snapshot fall on day 2-3; the time series work per month.
for c in ["valid_from", "valid_to"]:
    df[c] = df[c].dt.to_period("M").dt.to_timestamp()

current = df[df["valid_to"].isna()]
df.shape, current.shape

## 1. Charakteristika dat

Data pocházejí z Registru územní identifikace, adres a nemovitostí (RÚIAN), z výměnného formátu VFR
(měsíční snapshoty `OB_UKSH` po obcích, ČÚZK). Pipeline z nich skládá verzovanou historii
(SCD2): každá změna atributů objektu založí novou verzi a zánik objektu uzavře platnost poslední verze.
Analyzované území zahrnuje město Brno a 29 okolních obcí.

In [ ]:
first_snapshot = df["valid_from"].min()
n_objects = df["code"].nunique()
removed_objects = df.loc[df["end_reason"] == "removed", "code"].nunique()
brno_share = (current["municipality_name"] == "Brno").mean()

table_basic = pd.DataFrame(
    [
        ("Období snapshotů", f"{df['snapshot_date'].min():%m/%Y} – {df['snapshot_date'].max():%m/%Y}"),
        ("Počet měsíčních snapshotů", thousands(df["snapshot_date"].nunique())),
        ("Počet obcí", df["municipality_code"].nunique()),
        ("Počet verzí záznamů", thousands(len(df))),
        ("Počet unikátních stavebních objektů", thousands(n_objects)),
        ("z toho aktuálně platných", thousands(current["code"].nunique())),
        ("z toho zaniklých během období", thousands(removed_objects)),
        ("Podíl Brna na aktuálních objektech", pct(brno_share)),
        ("Medián počtu verzí na objekt", f"{df.groupby('code').size().median():.0f}"),
    ],
    columns=["Charakteristika", "Hodnota"],
)
table_basic.to_csv(TABLE_DIR / "tab_zakladni_charakteristiky.csv", index=False)
table_basic

### Použité atributy a jejich úplnost

Úplnost je počítaná nad aktuálními objekty. Atributy, které RÚIAN nevyžaduje (podlažnost, zastavěná plocha,
rok dokončení), jsou vyplněné jen částečně – s tím je potřeba počítat u všech následujících vizualizací.

In [ ]:
attributes = [
    ("usage_type_code", "Způsob využití", "CE_ZPUSOB_VYUZITI_OBJEKTU"),
    ("building_type_code", "Typ stavebního objektu", "CS_TYP_STAVEBNIHO_OBJEKTU"),
    ("construction_type_code", "Druh konstrukce", "CE_DRUH_KONSTRUKCE"),
    ("floor_count", "Počet podlaží", "–"),
    ("unit_count", "Počet bytů", "–"),
    ("built_up_area", "Zastavěná plocha (m²)", "–"),
    ("completion_date", "Datum dokončení", "–"),
    ("heating_type_code", "Způsob vytápění", "CE_ZPUSOB_VYTAPENI"),
    ("gas_connection_code", "Připojení na plyn", "CE_PRIPOJENI_PLYNU"),
    ("water_connection_code", "Připojení na vodovod", "CE_PRIPOJENI_VODY"),
    ("sewage_connection_code", "Připojení na kanalizaci", "CE_PRIPOJENI_KANAL"),
    ("elevator_code", "Vybavení výtahem", "CE_VYBAVENI_VYTAHEM"),
]
table_attrs = pd.DataFrame(
    {
        "Atribut": [a[1] for a in attributes],
        "Sloupec": [a[0] for a in attributes],
        "Číselník": [a[2] for a in attributes],
        "Vyplněno (%)": [round(100 * current[a[0]].notna().mean(), 1) for a in attributes],
    }
)
table_attrs.to_csv(TABLE_DIR / "tab_atributy.csv", index=False)
table_attrs

In [ ]:
fig, ax = plt.subplots(figsize=(6.3, 3.6))
t = table_attrs.sort_values("Vyplněno (%)")
ax.barh(t["Atribut"], t["Vyplněno (%)"], color=BLUE, height=0.65)
for y, v in enumerate(t["Vyplněno (%)"]):
    ax.text(v + 1, y, f"{v:.0f} %", va="center", color=INK_SOFT, fontsize=8)
ax.set_xlim(0, 105)
ax.set_xlabel("Podíl aktuálních objektů s vyplněnou hodnotou (%)")
ax.set_title("Úplnost atributů stavebních objektů")
ax.grid(axis="y", visible=False)
save(fig, "01_uplnost_atributu")
plt.show()

## 2. Struktura historie

Co historie obsahuje a jak se vyvíjí v čase. První snapshot (09/2015) je výchozí stav – všechny tehdy existující
objekty mají `valid_from` v tomto měsíci, proto se do časových řad událostí nezapočítává.

In [ ]:
per_object = df.groupby("code").size()
dist = per_object.clip(upper=6).value_counts().sort_index()
labels = [str(i) for i in dist.index[:-1]] + ["6 a více"]

fig, ax = plt.subplots(figsize=(6.3, 3.0))
ax.bar(labels, dist.values, color=BLUE, width=0.65)
for x, v in enumerate(dist.values):
    ax.text(x, v, pct(v / len(per_object)), ha="center", va="bottom", color=INK_SOFT, fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(thousands))
ax.set_xlabel("Počet verzí objektu")
ax.set_ylabel("Počet objektů")
ax.set_title("Kolik verzí má jeden stavební objekt")
ax.grid(axis="x", visible=False)
save(fig, "02_pocet_verzi_na_objekt")
plt.show()

In [ ]:
# A first version after the baseline is a new object, later ones are changes;
# 'removed' ends an object, 'change' ends a version replaced by the next one.
is_first = df["valid_from"] == df.groupby("code")["valid_from"].transform("min")
after_baseline = df["valid_from"] > first_snapshot
new_objects = df[after_baseline & is_first].groupby("valid_from").size()
changes = df[after_baseline & ~is_first].groupby("valid_from").size()
removals = df[df["end_reason"] == "removed"].groupby("valid_to").size()

months = pd.date_range(first_snapshot + pd.DateOffset(months=1), df["valid_from"].max(), freq="MS")
events = pd.DataFrame({
    "Nové objekty": new_objects, "Změny atributů": changes, "Zánik objektu": removals,
}).reindex(months).fillna(0)
events.describe().loc[["mean", "50%", "max"]].round(0)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(6.3, 5.4), sharex=True)
for ax, (name, color) in zip(axes, [("Změny atributů", BLUE), ("Nové objekty", AQUA), ("Zánik objektu", ORANGE)]):
    ax.bar(events.index, events[name], width=24, color=color)
    ax.set_title(name, fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(thousands))
    ax.margins(x=0.01)
axes[-1].set_xlabel("Měsíc snapshotu")
fig.supylabel("Počet objektů za měsíc", fontsize=9, color=INK_SOFT)
fig.tight_layout()
save(fig, "03_udalosti_v_case")
plt.show()

In [ ]:
# Valid versions at the end of each month: cumulative starts minus cumulative ends.
all_months = pd.date_range(first_snapshot, df["valid_from"].max(), freq="MS")
start_counts = df.groupby("valid_from").size().reindex(all_months, fill_value=0)
end_counts = df.groupby("valid_to").size().reindex(all_months, fill_value=0)
valid_objects = start_counts.cumsum() - end_counts.cumsum()

fig, ax = plt.subplots(figsize=(6.3, 3.0))
ax.plot(valid_objects.index, valid_objects.values, color=BLUE, linewidth=2)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(thousands))
ax.set_ylim(valid_objects.min() * 0.97, valid_objects.max() * 1.02)
ax.set_ylabel("Počet platných objektů")
ax.set_title("Vývoj počtu platných stavebních objektů")
save(fig, "04_pocet_platnych_objektu")
plt.show()
valid_objects.iloc[[0, -1]]

### Poznámky ke kapitole 1–2 (podklady pro text)

- Úplnost: typ objektu a způsob využití jsou vyplněné téměř vždy, ostatní atributy (podlažnost, konstrukce,
  přípojky, vytápění, rok dokončení) jen u cca 63–67 % aktuálních objektů; zastavěná plocha a počet bytů u 82 %, resp. 83 %.
- Historie: 29 % objektů má jedinou verzi, typický objekt (medián) dvě; 6 a více verzí má jen asi 1 %.
- Změny atributů jsou soustředěné do několika hromadných aktualizací – nejvýraznější 01/2024
  (43 521 verzí) a 02/2024 (18 041), dále 07/2022 (11 418) a 01/2016 (4 952). Měsíčně běžně stovky.
- Nových objektů přibývá stabilně 60–70 měsíčně (celkem 8 935 po výchozím stavu), zánik je nárazový:
  22 347 zaniklých objektů, největší vlny 04/2017 (2 351) a 02/2023 (2 306).
- Počet platných objektů klesá z 115 632 (09/2015) na 102 220 (08/2026), protože zániků je výrazně více než nových
  objektů. Pokles tvoří především několik skoků (2016–2017, 2022–2023), což odpovídá hromadnému čištění
  registru, ne reálnému úbytku zástavby – v textu je třeba na to upozornit.
- Chybějící snapshoty 12/2018, 03/2019, 03/2021 a 06/2021 (a poškozený 05/2016 u Brna) se v časové ose projeví tím,
  že se události dvou měsíců sloučí do následujícího.

## 3. Typ a způsob využití objektu

Od této kapitoly se pracuje jen s **aktuálním stavem** (verze bez `valid_to`), tedy s 102 220 objekty.
Názvy hodnot pocházejí z číselníků RÚIAN.

In [ ]:
def load_codelist(name):
    cl = pd.read_csv(CODELISTS / f"{name}.csv", sep=";", encoding="cp1250", dtype=str)
    return cl.set_index("KOD")["NAZEV"]


usage_names = load_codelist("CE_ZPUSOB_VYUZITI_OBJEKTU")
usage_names = usage_names.replace({
    "stavba pro rodinnou rekreaci": "rodinná rekreace",
    "stavba pro výrobu a skladování": "výroba a skladování",
    "stavba občanského vybavení": "občanská vybavenost",
    "stavba technického vybavení": "technické vybavení",
    "stavba pro administrativu": "administrativa",
    "stavba pro obchod": "obchod",
})
building_names = load_codelist("CS_TYP_STAVEBNIHO_OBJEKTU")

# Code 5 is the pre-2016 name of code 15 (civic amenities); older records were not converted.
current = current.assign(
    is_brno=current["municipality_name"] == "Brno",
    usage_type_code=current["usage_type_code"].replace({"5": "15"}),
)
current.shape

In [ ]:
type_share = current["building_type_code"].map(building_names).value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(6.3, 1.9))
t = type_share.sort_values()
ax.barh(t.index, t.values, color=BLUE, height=0.6)
for y, v in enumerate(t.values):
    ax.text(v + 1, y, f"{v:.1f} %", va="center", color=INK_SOFT, fontsize=8)
ax.set_xlim(0, 75)
ax.set_xlabel("Podíl aktuálních objektů (%)")
ax.set_title("Typ stavebního objektu")
ax.grid(axis="y", visible=False)
save(fig, "05_typ_objektu")
plt.show()
type_share.round(1)

In [ ]:
TOP_USAGES = 9
usage_counts = current["usage_type_code"].value_counts()
top_codes = list(usage_counts.index[:TOP_USAGES])
usage_group = current["usage_type_code"].where(current["usage_type_code"].isin(top_codes), "other")
usage_share = usage_group.value_counts(normalize=True) * 100
usage_share.index = [usage_names.get(c, "ostatní a neuvedeno") for c in usage_share.index]
usage_share = usage_share.sort_values()
other_label = "ostatní a neuvedeno"
usage_share = pd.concat([usage_share[[other_label]], usage_share.drop(other_label)])

fig, ax = plt.subplots(figsize=(6.3, 3.6))
colors = [GRAY if n == other_label else BLUE for n in usage_share.index]
ax.barh(usage_share.index, usage_share.values, color=colors, height=0.65)
for y, v in enumerate(usage_share.values):
    ax.text(v + 0.6, y, f"{v:.1f} %", va="center", color=INK_SOFT, fontsize=8)
ax.set_xlim(0, 55)
ax.set_xlabel("Podíl aktuálních objektů (%)")
ax.set_title("Způsob využití stavebního objektu")
ax.grid(axis="y", visible=False)
save(fig, "06_zpusob_vyuziti")
plt.show()
usage_share.sort_values(ascending=False).round(1)

### Brno a okolní obce

Srovnání podílů nejčastějších způsobů využití ve městě Brně a v ostatních 29 obcích. Každá skupina je normalizovaná
zvlášť, takže sloupce říkají, z čeho se zástavba skládá, ne kolik jí je.

In [ ]:
compare_codes = top_codes[:6]
share_by_area = (
    current.groupby("is_brno")["usage_type_code"]
    .value_counts(normalize=True)
    .mul(100)
    .unstack(0)
    .loc[compare_codes]
    .rename(columns={True: "Brno", False: "Okolní obce"})
)
share_by_area.index = share_by_area.index.map(usage_names)
share_by_area = share_by_area.iloc[::-1]

fig, ax = plt.subplots(figsize=(6.3, 3.4))
y = range(len(share_by_area))
h = 0.36
ax.barh([i + h / 2 for i in y], share_by_area["Brno"], height=h, color=BLUE, label="Brno")
ax.barh([i - h / 2 for i in y], share_by_area["Okolní obce"], height=h, color=ORANGE, label="Okolní obce")
for i, (b, o) in enumerate(zip(share_by_area["Brno"], share_by_area["Okolní obce"])):
    ax.text(b + 0.6, i + h / 2, f"{b:.1f} %", va="center", fontsize=7.5, color=INK_SOFT)
    ax.text(o + 0.6, i - h / 2, f"{o:.1f} %", va="center", fontsize=7.5, color=INK_SOFT)
ax.set_yticks(list(y), share_by_area.index)
ax.set_xlim(0, 65)
ax.set_xlabel("Podíl objektů v dané oblasti (%)")
ax.set_title("Způsob využití: Brno a okolní obce")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
save(fig, "07_vyuziti_brno_okoli")
plt.show()
share_by_area.round(1)

## 4. Podlažnost, velikost a počet bytů

Počet podlaží je vyplněný u 66 % aktuálních objektů, zastavěná plocha u 82 % a počet bytů u 83 %; grafy
vždy počítají jen z vyplněných hodnot.

In [ ]:
floors = current["floor_count"].dropna().astype(int)
floor_dist = floors.clip(upper=10).value_counts().sort_index()
floor_labels = [str(i) for i in floor_dist.index[:-1]] + ["10 a více"]

fig, ax = plt.subplots(figsize=(6.3, 3.0))
ax.bar(floor_labels, floor_dist.values, color=BLUE, width=0.65)
for x, v in enumerate(floor_dist.values):
    ax.text(x, v, pct(v / len(floors)), ha="center", va="bottom", color=INK_SOFT, fontsize=7.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(thousands))
ax.set_xlabel("Počet podlaží")
ax.set_ylabel("Počet objektů")
ax.set_title("Podlažnost stavebních objektů")
ax.grid(axis="x", visible=False)
save(fig, "08_podlaznost")
plt.show()
floors.describe().round(2)

In [ ]:
FLOOR_CLASSES = [(1, 1, "1"), (2, 2, "2"), (3, 3, "3"), (4, 5, "4–5"), (6, 99, "6 a více")]
RAMP = ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]

with_floors = current.dropna(subset=["floor_count"])
floor_usage_codes = [c for c in top_codes if c in set(with_floors["usage_type_code"])][:6]
subset = with_floors[with_floors["usage_type_code"].isin(floor_usage_codes)]
class_of = pd.cut(
    subset["floor_count"].astype(int),
    bins=[lo - 1 for lo, _, _ in FLOOR_CLASSES] + [99],
    labels=[lbl for _, _, lbl in FLOOR_CLASSES],
)
floor_by_usage = pd.crosstab(subset["usage_type_code"], class_of, normalize="index").mul(100)
floor_by_usage = floor_by_usage.loc[floor_usage_codes]
floor_by_usage.index = floor_by_usage.index.map(usage_names)
floor_by_usage = floor_by_usage.iloc[::-1]

fig, ax = plt.subplots(figsize=(6.3, 3.4))
left = pd.Series(0.0, index=floor_by_usage.index)
for color, col in zip(RAMP, floor_by_usage.columns):
    vals = floor_by_usage[col]
    ax.barh(floor_by_usage.index, vals, left=left, color=color, height=0.65,
            edgecolor="#fcfcfb", linewidth=1.2, label=col)
    for name, v, l in zip(floor_by_usage.index, vals, left):
        if v >= 7:
            ax.text(l + v / 2, name, f"{v:.0f}", ha="center", va="center", fontsize=7.5,
                    color="white" if color in RAMP[2:] else "#0b0b0b")
    left += vals
ax.set_xlim(0, 100)
ax.set_xlabel("Podíl objektů s vyplněnou podlažností (%)")
ax.set_title("Podlažnost podle způsobu využití")
ax.grid(axis="y", visible=False)
ax.legend(title="Počet podlaží", ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.2), fontsize=8,
          title_fontsize=8)
save(fig, "09_podlaznost_podle_vyuziti")
plt.show()
floor_by_usage.round(0)

In [ ]:
area = current["built_up_area"].dropna()
area = area[area > 0]
bins = np.logspace(np.log10(area.min()), np.log10(area.max()), 45)
median_area = area.median()

fig, ax = plt.subplots(figsize=(6.3, 3.0))
ax.hist(area, bins=bins, color=BLUE)
ax.set_xscale("log")
ax.axvline(median_area, color=ORANGE, linewidth=1.6)
ax.set_xlim(5, 20000)
ax.text(450, ax.get_ylim()[1] * 0.85, f"oranžová čára: medián {median_area:.0f} m²", color=INK_SOFT, fontsize=8)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: thousands(x)))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(thousands))
ax.set_xlabel("Zastavěná plocha (m², logaritmické měřítko)")
ax.set_ylabel("Počet objektů")
ax.set_title("Zastavěná plocha stavebních objektů")
save(fig, "10_zastavena_plocha")
plt.show()
area.describe(percentiles=[0.05, 0.5, 0.95, 0.99]).round(0)

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

area_usage = current[current["usage_type_code"].isin(top_codes[:7]) & (current["built_up_area"] > 0)]
order = list(top_codes[:7])
groups = [area_usage.loc[area_usage["usage_type_code"] == c, "built_up_area"].dropna() for c in order]
names = [f"{usage_names[c]} (n={thousands(len(g))})" for c, g in zip(order, groups)]

fig, ax = plt.subplots(figsize=(6.3, 3.9))
ax.boxplot(groups[::-1], vert=False, whis=(5, 95), showfliers=False, patch_artist=True,
           medianprops={"color": ORANGE, "linewidth": 1.8}, widths=0.6,
           boxprops={"facecolor": "#cde2fb", "edgecolor": BLUE},
           whiskerprops={"color": BLUE}, capprops={"color": BLUE})
ax.set_yticks(range(1, len(names) + 1), names[::-1])
ax.set_xscale("log")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: thousands(x)))
ax.set_xlabel("Zastavěná plocha (m², logaritmické měřítko)")
ax.set_title("Zastavěná plocha podle způsobu využití")
ax.grid(axis="y", visible=False)
ax.legend(
    handles=[
        Line2D([0], [0], color=ORANGE, linewidth=1.8, label="medián"),
        Patch(facecolor="#cde2fb", edgecolor=BLUE, label="střední polovina objektů (25.–75. percentil)"),
        Line2D([0], [0], color=BLUE, linewidth=1.2, label="rozsah 5.–95. percentil"),
    ],
    loc="upper center", bbox_to_anchor=(0.4, -0.18), ncol=3, fontsize=7.5, handlelength=1.6,
)
save(fig, "11_plocha_podle_vyuziti")
plt.show()

In [ ]:
apartment_houses = current.loc[current["usage_type_code"] == "6", "unit_count"].dropna().astype(int)
unit_bins = list(range(0, 61, 2))

fig, ax = plt.subplots(figsize=(6.3, 3.0))
ax.hist(apartment_houses.clip(upper=60), bins=unit_bins, color=BLUE)
ax.axvline(apartment_houses.median(), color=ORANGE, linewidth=1.6)
ax.text(38, ax.get_ylim()[1] * 0.8,
        f"oranžová čára: medián {apartment_houses.median():.0f} bytů", color=INK_SOFT, fontsize=8)
ax.set_xlabel("Počet bytů v bytovém domě (hodnoty nad 60 sloučeny)")
ax.set_ylabel("Počet bytových domů")
ax.set_title("Počet bytů v bytových domech")
save(fig, "12_pocet_bytu_bytove_domy")
plt.show()
apartment_houses.describe().round(1)

In [ ]:
def describe_column(series, label):
    s = series.dropna()
    return {
        "Veličina": label, "Počet hodnot": len(s), "Medián": s.median(), "Dolní kvartil": s.quantile(0.25),
        "Horní kvartil": s.quantile(0.75), "95. percentil": s.quantile(0.95), "Maximum": s.max(),
    }


table_numeric = pd.DataFrame([
    describe_column(current["floor_count"].astype(float), "Počet podlaží"),
    describe_column(current.loc[current["built_up_area"] > 0, "built_up_area"], "Zastavěná plocha (m²)"),
    describe_column(current["unit_count"].astype(float), "Počet bytů (všechny objekty)"),
    describe_column(apartment_houses.astype(float), "Počet bytů (bytové domy)"),
]).round(0)
table_numeric.to_csv(TABLE_DIR / "tab_ciselne_atributy.csv", index=False)
table_numeric

### Poznámky ke kapitolám 3–4 (podklady pro text)

- Typ objektu: 60,8 % budovy s číslem popisným, 19,0 % s číslem evidenčním, 20,2 % bez čísla.
- Využití: rodinný dům 45,8 %, garáž 17,4 %, rodinná rekreace 11,0 %, bytový dům 7,6 %, jiná stavba 6,0 %; zbytek pod 3 % každý (občanská vybavenost 2,7 % po sloučení kódů 5 a 15).
- Brno × okolí: v Brně 9,9 % bytových domů (v okolí 1,7 %) a 19,0 % garáží (13,8 %); v okolí 56,8 % rodinných domů (v Brně 41,9 %).
- Podlažnost (66 % objektů): 1 podlaží 34 %, 2 podlaží 40 %, 3 podlaží 13 %, 6 a více jen 5 %; maximum 32 podlaží.
- Bytové domy: 85 % má 4 a více podlaží (48 % 4–5, 37 % 6 a více); rodinné domy nejčastěji 2 podlaží (52 %); garáže 94 % jednopodlažní.
- Zastavěná plocha: medián 105 m², 5.–95. percentil 18–541 m², bimodální rozdělení (druhý vrchol kolem 20 m² = garáže a rekreace).
  Mediány podle využití: garáž 20 m², rekreace 29 m², rodinný dům 110 m², bytový dům 298 m², občanská vybavenost 431 m², výroba 692 m².
- Počet bytů v bytových domech: medián 15, horní kvartil 25, 4 % domů má více než 60 bytů, maximum 346.
  Počet bytů u všech objektů má medián 1.

## 5. Stáří zástavby a vývoj v čase

Rok dokončení má vyplněný 63 % aktuálních objektů. Je nutné počítat s tím, že u starších budov jde často o odhad:
64 % všech dat dokončení připadá na 31. 12. a ve statistice vidíme výrazné špičky na hranicích 1919, 1945, 1960, 1970,
1980, 1990 a 2000. Odpovídají obdobím, do kterých se starší zástavba v registru zařazuje, a ne skutečným rokům výstavby.
Přesné roky jsou věrohodné až od roku 2001, proto se dřívější zástavba hodnotí po obdobích a tempo výstavby po letech od roku 2001.

In [ ]:
dated = current.dropna(subset=["completion_date"]).copy()
dated["year"] = dated["completion_date"].dt.year
share_dec31 = ((dated["completion_date"].dt.month == 12) & (dated["completion_date"].dt.day == 31)).mean()
print(f"s datem dokončení: {len(dated)} ({len(dated) / len(current):.1%}), 31. 12.: {share_dec31:.1%}")

PERIOD_EDGES = [0, 1919, 1945, 1960, 1970, 1980, 1990, 2000, 2010, 2100]
PERIOD_LABELS = ["do 1919", "1920–1945", "1946–1960", "1961–1970", "1971–1980", "1981–1990",
                 "1991–2000", "2001–2010", "od 2011"]
dated["period"] = pd.cut(dated["year"], bins=PERIOD_EDGES, labels=PERIOD_LABELS)
period_counts = dated["period"].value_counts().reindex(PERIOD_LABELS)

fig, ax = plt.subplots(figsize=(6.3, 3.2))
ax.bar(PERIOD_LABELS, period_counts.values, color=BLUE, width=0.7)
for x, v in enumerate(period_counts.values):
    ax.text(x, v, thousands(v), ha="center", va="bottom", fontsize=7.5, color=INK_SOFT)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(thousands))
ax.set_ylabel("Počet objektů")
ax.set_title("Stáří zástavby podle období dokončení")
ax.tick_params(axis="x", labelsize=7.5)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
ax.grid(axis="x", visible=False)
save(fig, "18_stari_zastavby")
plt.show()
period_counts

### Tempo výstavby

Počet objektů podle roku dokončení (od roku 2001, kdy jsou data dokončení přesná), členěný podle způsobu využití.
Zahrnuje jen objekty, které v registru zůstaly – zaniklé objekty jsou z grafu vyloučené. Rok 2026 není dokončený a rok 2025
je pravděpodobně dosud neúplný (kolaudace se do registru zapisují se zpožděním), proto jsou oba vyznačeny jinak.

In [ ]:
rate = dated[(dated["year"] >= 2001) & (dated["year"] <= 2026)].copy()
GROUPS = [("7", "rodinný dům"), ("6", "bytový dům"), ("18", "garáž")]
group_of = {code: label for code, label in GROUPS}
rate["group"] = rate["usage_type_code"].map(group_of).fillna("ostatní")
by_year = rate.groupby(["year", "group"]).size().unstack(fill_value=0)[[g for _, g in GROUPS] + ["ostatní"]]

STACK_COLORS = [BLUE, ORANGE, AQUA, GRAY]
fig, ax = plt.subplots(figsize=(6.3, 3.4))
bottom = np.zeros(len(by_year))
for color, col in zip(STACK_COLORS, by_year.columns):
    ax.bar(by_year.index, by_year[col], bottom=bottom, color=color, width=0.75, label=col,
           edgecolor="#fcfcfb", linewidth=0.6)
    bottom += by_year[col].values
for year in (2025, 2026):
    ax.text(year, bottom[list(by_year.index).index(year)] + 15, "neúplné", ha="center", fontsize=6.5,
            color=INK_SOFT, rotation=90, va="bottom")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(thousands))
ax.set_ylabel("Počet dokončených objektů")
ax.set_title("Tempo výstavby podle roku dokončení")
ax.grid(axis="x", visible=False)
ax.legend(loc="upper right", ncol=2, fontsize=8)
save(fig, "19_tempo_vystavby")
plt.show()
by_year.sum(axis=1).loc[2001:2024].describe().round(0), by_year.loc[2024]

In [ ]:
print("průměr 2012–2024:", by_year.loc[2012:2024].sum(axis=1).mean().round(0),
      "| medián počtu nových objektů v historii za rok:",
      round(df[(df["valid_from"] > first_snapshot) & (df["valid_from"] == df.groupby("code")["valid_from"].transform("min"))]
            .groupby(df["valid_from"].dt.year).size().loc[2016:2025].median()))

### Konstrukce podle stáří

Druh konstrukce (číselník má i podrobné podskupiny od roku 2016, sloučeno do pěti skupin) ve vztahu k období dokončení.

In [ ]:
def construction_group(code):
    if code in ("1", "2", "3", "10"):
        return "zdivo (cihly, kámen)"
    if code in ("4", "41", "42", "43"):
        return "stěnové panely"
    if code == "6" or code.startswith("6"):
        return "dřevo"
    if code == "11":
        return "monolit"
    if code in ("5", "7"):
        return "jiné"
    return "nezjištěno"


with_construction = dated.dropna(subset=["construction_type_code"]).copy()
with_construction["construction"] = with_construction["construction_type_code"].map(construction_group)
CONSTRUCTION_ORDER = ["zdivo (cihly, kámen)", "stěnové panely", "dřevo", "monolit", "jiné", "nezjištěno"]
construction_by_period = (
    pd.crosstab(with_construction["period"], with_construction["construction"], normalize="index")
    .mul(100).reindex(columns=CONSTRUCTION_ORDER, fill_value=0).reindex(PERIOD_LABELS)
)

fig, ax = plt.subplots(figsize=(6.3, 3.6))
bottom = np.zeros(len(construction_by_period))
colors = [BLUE, ORANGE, AQUA, "#eda100", "#e87ba4", GRAY]
for color, col in zip(colors, CONSTRUCTION_ORDER):
    vals = construction_by_period[col].values
    ax.bar(PERIOD_LABELS, vals, bottom=bottom, color=color, width=0.7, label=col,
           edgecolor="#fcfcfb", linewidth=0.8)
    for x, (v, b) in enumerate(zip(vals, bottom)):
        if v >= 8:
            ax.text(x, b + v / 2, f"{v:.0f}", ha="center", va="center", fontsize=7, color="white" if col != "jiné" else "black")
    bottom += vals
ax.set_ylim(0, 100)
ax.set_ylabel("Podíl objektů (%)")
ax.set_title("Druh konstrukce podle období dokončení")
ax.tick_params(axis="x", labelsize=7.5)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
ax.grid(axis="x", visible=False)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.28), fontsize=7.5)
save(fig, "20_konstrukce_podle_obdobi")
plt.show()
construction_by_period.round(0)

### Životnost objektů

Zánik objektu v registru nemusí znamenat zbourání – část se do historie dostane jako vyčištění neúplných záznamů.
Z historie proto nejprve ověříme, co o zaniklých objektech víme; ukáže se, že životnost objektů z nich odvodit nelze.

In [ ]:
last_version = df.sort_values("valid_from").groupby("code").tail(1)
removed = last_version[last_version["end_reason"] == "removed"]
still_valid = last_version[last_version["valid_to"].isna()]

CHECKED = [("usage_type_code", "způsob využití"), ("built_up_area", "zastavěná plocha"),
           ("floor_count", "počet podlaží"), ("completion_date", "datum dokončení")]
filled = pd.DataFrame({
    "Zaniklé objekty": [removed[c].notna().mean() * 100 for c, _ in CHECKED],
    "Aktuální objekty": [still_valid[c].notna().mean() * 100 for c, _ in CHECKED],
}, index=[label for _, label in CHECKED])

fig, ax = plt.subplots(figsize=(6.3, 3.0))
y = np.arange(len(filled))
h = 0.36
ax.barh(y + h / 2, filled["Aktuální objekty"], height=h, color=BLUE, label="aktuální objekty")
ax.barh(y - h / 2, filled["Zaniklé objekty"], height=h, color=ORANGE, label="zaniklé objekty")
for i, (a, z) in enumerate(zip(filled["Aktuální objekty"], filled["Zaniklé objekty"])):
    ax.text(a + 1, i + h / 2, f"{a:.0f} %", va="center", fontsize=7.5, color=INK_SOFT)
    ax.text(z + 1, i - h / 2, f"{z:.0f} %", va="center", fontsize=7.5, color=INK_SOFT)
ax.set_yticks(y, filled.index)
ax.invert_yaxis()
ax.set_xlim(0, 112)
ax.set_xlabel("Podíl objektů s vyplněným údajem (%)")
ax.set_title("Co víme o zaniklých objektech")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
save(fig, "21_zanikle_objekty_vyplnenost")
plt.show()
removed_building_types = removed["building_type_code"].map(building_names).value_counts(normalize=True).mul(100)
filled.round(0), removed_building_types.round(1), len(removed)

### Poznámky ke kapitole 5 (podklady pro text)

- Rok dokončení má 63,2 % aktuálních objektů; 64,4 % dat je 31. 12. a starší data se hromadí na hranicích období
  (1919, 1945, 1960, 1970, 1980, 1990, 2000) – jde o zařazení do období, ne přesný rok. Přesné roky jsou věrohodné od 2001.
- Stáří: 27 % objektů s datem je z doby do roku 1945, 32 % do roku 1960, 34 % z let 2001 a novější.
  Největší období je 1920–1945 (12 855), od 2011 (12 028) a 2001–2010 (10 014).
- Tempo výstavby 2001–2024: špička 2007 (1 429), 2004 (1 227), 2005 (1 069); minimum 2012 (648). Průměr 2001–2010 ≈ 1 000 objektů ročně,
  2012–2024 ≈ 795 ročně. Rodinné domy tvoří 64 % z nich, bytové domy a garáže po 7 %. Rok 2025 (439) a 2026 (208) jsou neúplné.
  Shoda s historií: medián nových objektů v historii za rok 2016–2025 je 808, tedy obdobně jako v datech dokončení.
- Konstrukce: ve všech obdobích dominuje zdivo (76–93 %). Stěnové panely jsou typické pro 1961–1990 (10 %, 17 %, 16 %), jinde do 3 %;
  79 % panelových objektů je z let 1961–1990. Dřevo roste v nejnovějším období (cca 5 %).
- Životnost: z historie nelze spolehlivě odvodit. Z 22 347 zaniklých objektů má využití vyplněné 32 %, zastavěnou plochu 3 %, podlažnost 6 %
  a datum dokončení 3 % (765 objektů); 71 % jsou budovy s číslem evidenčním. Většinou jde o neúplné záznamy odstraněné z registru,
  ne o zbourané budovy. Stáří při zániku (medián 15 let, n=765) zkreslují zástupná data (např. 31. 12. 2000).

## 6. Technická vybavenost

Přípojky na sítě (voda, kanalizace, plyn), způsob vytápění a výtah. Stejně jako dřív jde o aktuální objekty.
Všechny tyto atributy jsou v RÚIAN vyplněné společně u přibližně dvou třetin objektů – nejdřív proto zjistíme, o které objekty jde.

In [ ]:
equipment_cols = [
    "water_connection_code", "sewage_connection_code", "gas_connection_code",
    "elevator_code", "heating_type_code",
]
current = current.assign(has_equipment=current[equipment_cols].notna().any(axis=1))
print(f"vyplněno alespoň jedno: {current['has_equipment'].mean():.1%}, "
      f"všech pět: {current[equipment_cols].notna().all(axis=1).mean():.1%}")

equipment_by_usage = (
    current[current["usage_type_code"].isin(top_codes[:7])]
    .groupby("usage_type_code")["has_equipment"].mean().mul(100)
)
equipment_by_usage.index = equipment_by_usage.index.map(usage_names)
equipment_by_usage = equipment_by_usage.sort_values()

fig, ax = plt.subplots(figsize=(6.3, 3.2))
ax.barh(equipment_by_usage.index, equipment_by_usage.values, color=BLUE, height=0.65)
for y, v in enumerate(equipment_by_usage.values):
    ax.text(v + 1, y, f"{v:.0f} %", va="center", color=INK_SOFT, fontsize=8)
ax.set_xlim(0, 110)
ax.set_xlabel("Podíl objektů s vyplněnými údaji o vybavení (%)")
ax.set_title("Kde jsou údaje o technické vybavenosti vyplněné")
ax.grid(axis="y", visible=False)
save(fig, "13_vybaveni_vyplnenost_podle_vyuziti")
plt.show()
equipment_by_usage.round(0)

### Přípojky na vodovod, kanalizaci a plyn

Podíly jsou počítané z objektů, u kterých je údaj vyplněný (včetně hodnoty „nezjištěno“, která je vyznačena šedě).

In [ ]:
WATER = {"1": "s vodovodem", "2": "bez vodovodu", "9": "nezjištěno"}
SEWAGE = {"1": "kanalizační síť", "2": "vlastní čistička", "3": "žumpa, jímka, septik",
          "4": "bez kanalizace a jímky", "8": "nezjištěno", "9": "nezjištěno"}
GAS = {"1": "plyn z veřejné sítě", "3": "bez plynu", "8": "nezjištěno", "9": "nezjištěno"}
GAS_OTHER = "zásobník a jiný plyn"


def category_shares(series, mapping, default=None):
    labels = series.dropna().map(lambda c: mapping.get(c, default))
    return labels.value_counts(normalize=True).mul(100)


panels = [
    ("Připojení na vodovod", category_shares(current["water_connection_code"], WATER)),
    ("Odvod odpadních vod", category_shares(current["sewage_connection_code"], SEWAGE)),
    ("Plyn", category_shares(current["gas_connection_code"], GAS, default=GAS_OTHER)),
]

fig, axes = plt.subplots(3, 1, figsize=(6.3, 5.0))
for ax, (title, shares) in zip(axes, panels):
    shares = shares.sort_values()
    known = [lbl for lbl in shares.index if lbl != "nezjištěno"]
    ordered = (["nezjištěno"] if "nezjištěno" in shares.index else []) + known
    shares = shares[ordered]
    ax.barh(shares.index, shares.values, height=0.65,
            color=[GRAY if lbl == "nezjištěno" else BLUE for lbl in shares.index])
    for y, v in enumerate(shares.values):
        ax.text(v + 1, y, f"{v:.1f} %", va="center", color=INK_SOFT, fontsize=8)
    ax.set_xlim(0, 100)
    ax.set_title(title, fontsize=9)
    ax.grid(axis="y", visible=False)
axes[-1].set_xlabel("Podíl objektů s vyplněným údajem (%)")
fig.tight_layout()
save(fig, "14_pripojky_voda_kanalizace_plyn")
plt.show()
[(t, s.round(1).to_dict()) for t, s in panels]

### Brno a okolní obce

Podíl objektů s přípojkou mezi objekty se zjištěným údajem (bez hodnot „nezjištěno“ a „nedefinováno“).

In [ ]:
def connected_share(series, yes_code, unknown=("8", "9")):
    known = current.loc[series.notna() & ~series.isin(unknown)]
    return known.assign(flag=series[known.index] == yes_code).groupby("is_brno")["flag"].mean().mul(100)


connected = pd.DataFrame({
    "vodovod": connected_share(current["water_connection_code"], "1"),
    "kanalizační síť": connected_share(current["sewage_connection_code"], "1"),
    "plyn z veřejné sítě": connected_share(current["gas_connection_code"], "1"),
    "výtah": connected_share(current["elevator_code"], "1"),
}).rename(index={True: "Brno", False: "Okolní obce"}).T

fig, ax = plt.subplots(figsize=(6.3, 3.0))
x = np.arange(len(connected))
w = 0.36
ax.bar(x - w / 2, connected["Brno"], width=w, color=BLUE, label="Brno")
ax.bar(x + w / 2, connected["Okolní obce"], width=w, color=ORANGE, label="Okolní obce")
for i, (b, o) in enumerate(zip(connected["Brno"], connected["Okolní obce"])):
    ax.text(i - w / 2, b + 1.5, f"{b:.0f} %", ha="center", fontsize=7.5, color=INK_SOFT)
    ax.text(i + w / 2, o + 1.5, f"{o:.0f} %", ha="center", fontsize=7.5, color=INK_SOFT)
ax.set_xticks(x, connected.index)
ax.set_ylim(0, 105)
ax.set_ylabel("Podíl objektů (%)")
ax.set_title("Vybavení objektů: Brno a okolní obce")
ax.grid(axis="x", visible=False)
ax.legend(loc="upper right", ncol=2)
save(fig, "15_vybaveni_brno_okoli")
plt.show()
connected.round(1)

### Způsob vytápění

Číselník vytápění má 39 hodnot: druh vytápění (domovní / dálkové / lokální) a od roku 2024 i zdroj energie.
Starší záznamy mají jen druh bez zdroje, proto zdroj známe jen u zlomku objektů.

In [ ]:
HEATING_KIND = {"1": "centrální domovní", "2": "centrální dálkové", "3": "lokální", "4": "jiné (i bez topení)",
                "9": "nezjištěno"}
HEATING_KIND_BY_TENS = {"1": "centrální domovní", "2": "centrální dálkové", "3": "lokální"}
FUEL = {"1": "zemní plyn", "2": "uhlí", "3": "dřevo a biomasa", "4": "pelety", "5": "elektřina",
        "6": "tepelné čerpadlo", "7": "solární systémy", "8": "kapalná paliva", "9": "jiné"}

heating = current["heating_type_code"].dropna()
kind = heating.map(lambda c: HEATING_KIND[c] if len(c) == 1 else HEATING_KIND_BY_TENS[c[0]])
kind_share = kind.value_counts(normalize=True).mul(100)
with_fuel = heating[heating.str.len() == 2]
fuel_share = with_fuel.str[1].map(FUEL).value_counts(normalize=True).mul(100)

fig, (ax_kind, ax_fuel) = plt.subplots(1, 2, figsize=(6.3, 3.2), gridspec_kw={"width_ratios": [1, 1]})
for ax, shares, title, color in [
    (ax_kind, kind_share.sort_values(), "Druh vytápění", BLUE),
    (ax_fuel, fuel_share.sort_values(), f"Zdroj energie (n={thousands(len(with_fuel))})", AQUA),
]:
    ax.barh(shares.index, shares.values, color=color, height=0.65)
    for y, v in enumerate(shares.values):
        ax.text(v + 1, y, f"{v:.0f} %" if v >= 1 else "< 1 %", va="center", color=INK_SOFT, fontsize=7.5)
    ax.set_xlim(0, 100)
    ax.set_title(title, fontsize=9)
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(axis="y", visible=False)
fig.supxlabel("Podíl objektů s vyplněným údajem (%)", fontsize=8, color=INK_SOFT)
fig.tight_layout()
save(fig, "16_zpusob_vytapeni")
plt.show()
kind_share.round(1), fuel_share.round(1)

### Výtah

Výtah dává smysl hlavně u vícepodlažních budov, proto ho srovnáváme s počtem podlaží.

In [ ]:
elevator = current[current["elevator_code"].isin(["1", "2"]) & current["floor_count"].notna()].copy()
elevator["has_elevator"] = elevator["elevator_code"] == "1"
elevator["floors"] = pd.cut(elevator["floor_count"].astype(int), bins=[0, 1, 2, 3, 4, 5, 7, 99],
                            labels=["1", "2", "3", "4", "5", "6–7", "8 a více"])
by_floors = elevator.groupby("floors", observed=True)["has_elevator"].mean().mul(100)
fig, ax = plt.subplots(figsize=(6.3, 3.0))
ax.bar(by_floors.index.astype(str), by_floors.values, width=0.65, color=BLUE)
for x, v in enumerate(by_floors.values):
    ax.text(x, v + 1.5, f"{v:.0f} %", ha="center", fontsize=8, color=INK_SOFT)
ax.set_xlabel("Počet podlaží")
ax.set_ylabel("Podíl objektů s výtahem (%)")
ax.set_ylim(0, 108)
ax.set_title("Výtah podle počtu podlaží")
ax.grid(axis="x", visible=False)
save(fig, "17_vytah_podle_podlazi")
plt.show()
by_floors.round(0)

### Poznámky ke kapitole 6 (podklady pro text)

- Údaje o vybavení jsou vyplněné společně: alespoň jeden z pěti atributů u 68,1 % objektů, všech pět u 64,9 %.
  Rodinné a bytové domy je mají vyplněné ze 100 %, občanská vybavenost 68 %, výroba 43 %, jiná stavba 40 %, rekreace 23 %, garáže 19 %.
- Vodovod: 86,0 % s vodovodem, 8,8 % bez, 5,2 % nezjištěno.
- Odpadní vody: 78,5 % kanalizační síť, 9,5 % žumpa/jímka/septik, 7,0 % bez kanalizace a jímky, 0,4 % vlastní čistička, 4,5 % nezjištěno.
- Plyn: 76,7 % veřejná síť, 17,9 % bez plynu, 5,2 % nezjištěno (zásobníkový plyn 0,1 %).
- Brno × okolí (mezi objekty se zjištěným údajem): vodovod 92 % × 87 %, kanalizační síť 87 % × 71 %, plyn z veřejné sítě 85 % × 72 %.
- Vytápění: 62,5 % centrální domovní, 15,5 % lokální, 11,5 % jiné (i bez topení), 5,5 % dálkové, 4,9 % nezjištěno.
  Zdroj energie je znám jen u 5 734 objektů (nový číselník od 2024): zemní plyn 54 %, elektřina 24 %, tepelné čerpadlo 14 %.
- Výtah: 3,8 % aktuálních objektů s výtahem (zjištěné údaje: 7,8 % v Brně, 1,0 % v okolí). Podíl roste s podlažností:
  do 3 podlaží ≤ 2 %, 4 podlaží 9 %, 5 podlaží 20 %, 6–7 podlaží 69 %, 8 a více 97 %.

## 7. Prostorový rozměr

Mapy vycházejí z definičních bodů objektů (`reference_point`), které mají vyplněné téměř všechny objekty (99,9 %);
obrysy budov má jen 97,8 %. Hustota a průměry se proto počítají v pravidelné šestiúhelníkové síti (cca 250 m)
a ne na obrysech. Souřadnice jsou přepočtené na metry od středu území (sever je nahoře).

In [ ]:
geo = current.dropna(subset=["reference_point_wgs84"]).copy()
points = shapely.from_wkb(geo["reference_point_wgs84"].values)
lon, lat = shapely.get_x(points), shapely.get_y(points)
lon0, lat0 = np.median(lon), np.median(lat)
geo["x"] = (lon - lon0) * np.cos(np.radians(lat0)) * 111_320
geo["y"] = (lat - lat0) * 111_320

SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
LABELLED = ["Brno", "Kuřim", "Šlapanice", "Modřice", "Bílovice nad Svitavou", "Střelice"]
SHORT_NAMES = {"Bílovice nad Svitavou": "Bílovice n. S."}
centres = geo.groupby("municipality_name")[["x", "y"]].median()
EXTENT = (geo["x"].min() - 500, geo["x"].max() + 500, geo["y"].min() - 500, geo["y"].max() + 500)
GRID = 110


def style_map(ax, labels=True):
    ax.set_xlim(EXTENT[0], EXTENT[1])
    ax.set_ylim(EXTENT[2], EXTENT[3])
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_visible(False)
    x0, y0 = EXTENT[0] + 1500, EXTENT[2] + 1500
    ax.plot([x0, x0 + 5000], [y0, y0], color=INK_SOFT, linewidth=2, solid_capstyle="butt")
    ax.text(x0 + 2500, y0 + 400, "5 km", ha="center", fontsize=7.5, color=INK_SOFT)
    if labels:
        for name in LABELLED:
            cx, cy = centres.loc[name]
            ax.annotate(SHORT_NAMES.get(name, name), (cx, cy), fontsize=7, color="#0b0b0b", ha="center",
                        va="center", bbox={"boxstyle": "round,pad=0.15", "fc": "white", "ec": "none", "alpha": 0.75})


geo[["x", "y"]].describe().round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(6.3, 5.2))
hb = ax.hexbin(geo["x"], geo["y"], gridsize=GRID, bins="log", mincnt=1, cmap=SEQ, linewidths=0.1,
               extent=EXTENT)
style_map(ax)
cb = fig.colorbar(hb, ax=ax, shrink=0.7, pad=0.01)
cb.set_label("Počet objektů v buňce (log. měřítko)", fontsize=8)
cb.ax.tick_params(labelsize=7)
ax.set_title("Hustota stavebních objektů")
save(fig, "23_mapa_hustota")
plt.show()

### Podlažnost a stáří v prostoru

Průměrná podlažnost a medián roku dokončení v buňkách sítě. Buňky s méně než 10 objekty s vyplněnou hodnotou se nevykreslují.

In [ ]:
floor_geo = geo.dropna(subset=["floor_count"])
fig, ax = plt.subplots(figsize=(6.3, 5.2))
hb = ax.hexbin(floor_geo["x"], floor_geo["y"], C=floor_geo["floor_count"].astype(float), gridsize=GRID,
               reduce_C_function=np.mean, mincnt=10, cmap=SEQ, extent=EXTENT, linewidths=0.1, vmin=1, vmax=6)
style_map(ax)
cb = fig.colorbar(hb, ax=ax, shrink=0.7, pad=0.01, extend="max")
cb.set_label("Průměrný počet podlaží", fontsize=8)
cb.ax.tick_params(labelsize=7)
ax.set_title("Průměrná podlažnost objektů")
save(fig, "24_mapa_podlaznost")
plt.show()
floor_geo["floor_count"].astype(float).groupby(geo["municipality_name"]).mean().loc[LABELLED].round(2)

In [ ]:
year_geo = geo.dropna(subset=["completion_date"]).assign(year=lambda f: f["completion_date"].dt.year)
fig, ax = plt.subplots(figsize=(6.3, 5.2))
hb = ax.hexbin(year_geo["x"], year_geo["y"], C=year_geo["year"], gridsize=GRID,
               reduce_C_function=np.median, mincnt=10, cmap=SEQ.reversed(), extent=EXTENT, linewidths=0.1,
               vmin=1920, vmax=2020)
style_map(ax)
cb = fig.colorbar(hb, ax=ax, shrink=0.7, pad=0.01)
cb.set_label("Medián roku dokončení", fontsize=8)
cb.ax.tick_params(labelsize=7)
ax.set_title("Stáří zástavby: medián roku dokončení")
save(fig, "25_mapa_rok_dokonceni")
plt.show()
year_geo.groupby("municipality_name")["year"].median().loc[LABELLED]

### Nová výstavba

Objekty dokončené v letech 2015–2024 (poslední úplné roky) nad pozadím všech objektů.

In [ ]:
RECENT = (2015, 2024)
new_geo = year_geo[year_geo["year"].between(*RECENT)]

fig, ax = plt.subplots(figsize=(6.3, 5.2))
ax.scatter(geo["x"], geo["y"], s=0.4, color="#cfcec9", linewidths=0, rasterized=True)
ax.scatter(new_geo["x"], new_geo["y"], s=2.5, color=ORANGE, linewidths=0, rasterized=True)
style_map(ax)
ax.set_title(f"Nová výstavba {RECENT[0]}–{RECENT[1]} (n={thousands(len(new_geo))})")
save(fig, "26_mapa_nova_vystavba")
plt.show()

In [ ]:
by_municipality = pd.DataFrame({
    "objects": current.groupby("municipality_name").size(),
    "new": current[current["completion_date"].dt.year.between(*RECENT)].groupby("municipality_name").size(),
}).fillna(0)
by_municipality["per_1000"] = by_municipality["new"] / by_municipality["objects"] * 1000
largest = by_municipality.sort_values("objects", ascending=False).head(12).sort_values("per_1000")

fig, ax = plt.subplots(figsize=(6.3, 3.6))
ax.barh(largest.index, largest["per_1000"], color=[ORANGE if n == "Brno" else BLUE for n in largest.index], height=0.65)
for y, (v, n) in enumerate(zip(largest["per_1000"], largest["new"])):
    ax.text(v + 1, y, f"{v:.0f} ({thousands(n)})", va="center", fontsize=7.5, color=INK_SOFT)
ax.set_xlim(0, largest["per_1000"].max() * 1.2)
ax.set_xlabel(f"Objekty dokončené {RECENT[0]}–{RECENT[1]} na 1\u00a0000 objektů (v závorce počet)")
ax.set_title("Nová výstavba v největších obcích")
ax.grid(axis="y", visible=False)
save(fig, "27_nova_vystavba_podle_obci")
plt.show()
largest.sort_values("per_1000", ascending=False).round(0)

### Poznámky ke kapitole 7 (podklady pro text)

- Prostorová data: definiční bod má 99,9 % objektů, obrys 97,8 %; mapy ukazují hustotu v síti ~250 m.
- Hustota: souvislé jádro v Brně, od něj paprsky podél údolí a silnic; okolní obce jsou izolované shluky (Kuřim, Šlapanice, Modřice, Bílovice n. S., Střelice).
- Podlažnost: průměr 2,5 podlaží v Brně a 1,75 v okolních obcích; nejvyšší v centru a v panelových sídlištích (4–6+), okolí je nízké (1–2).
- Stáří: medián roku dokončení je 1980 v Brně a 2001 v okolních obcích; nejstarší zástavba je v jádru Brna (okolo 1920–1950), nejnovější v Kuřimi a Modřicích (2004).
- Nová výstavba 2015–2024: 8 197 objektů, z toho 4 689 v Brně. Na 1 000 stávajících objektů je to 63 v Brně a 125 v okolních obcích (nejvíc Moravany 163,
  Střelice 159, Popůvky 153; Šlapanice 62). Nová výstavba je rozptýlená po celém území, výrazné shluky jsou na okrajích Brna.

## 9. Příklady navržených vizualizací

Ilustrační náhledy vizualizací z tabulky navržených pohledů v aplikaci, vytvořené z reálných dat. V aplikaci budou interaktivní;
zde jde o statické ukázky, které ukazují, jaká data a jakou podobu vizualizace mají. Hustota v síti (příklad 2) je mapa z kapitoly 7
(obrázek `23_mapa_hustota`).

In [ ]:
import geopandas as gpd

hist = pd.read_parquet(DATA, columns=[
    "code", "valid_from", "valid_to", "usage_type_code", "floor_count", "unit_count", "built_up_area",
    "geometry_wgs84", "reference_point_wgs84",
])
for c in ["valid_from", "valid_to"]:
    hist[c] = pd.to_datetime(hist[c]).dt.to_period("M").dt.to_timestamp()
hist = hist.dropna(subset=["reference_point_wgs84"]).reset_index(drop=True)
_pts = shapely.from_wkb(hist["reference_point_wgs84"].values)
hist["x"] = (shapely.get_x(_pts) - lon0) * np.cos(np.radians(lat0)) * 111_320
hist["y"] = (shapely.get_y(_pts) - lat0) * 111_320


def in_window(frame, window):
    x0, x1, y0, y1 = window
    return frame[frame["x"].between(x0, x1) & frame["y"].between(y0, y1)]


def local_geometry(frame):
    geoms = shapely.from_wkb(frame["geometry_wgs84"].values)
    return shapely.transform(
        geoms,
        lambda a: np.column_stack([(a[:, 0] - lon0) * np.cos(np.radians(lat0)) * 111_320, (a[:, 1] - lat0) * 111_320]),
    )


def draw_objects(ax, frame, colors, window):
    has_shape = frame["geometry_wgs84"].notna().values
    shapes = gpd.GeoSeries(local_geometry(frame[has_shape]))
    shapes.plot(ax=ax, color=np.array(colors)[has_shape], linewidth=0)
    rest = frame[~has_shape]
    ax.scatter(rest["x"], rest["y"], s=6, color=np.array(colors)[~has_shape], linewidths=0)
    ax.set_xlim(window[0], window[1])
    ax.set_ylim(window[2], window[3])
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_visible(False)


hist.shape

### Příklad 1 – mapa objektů s barvou podle atributu

In [ ]:
WINDOW_MAP = (1700, 2500, -100, 700)
USAGE_COLORS = {"7": BLUE, "6": ORANGE, "18": AQUA}
USAGE_LABELS = {"7": "rodinný dům", "6": "bytový dům", "18": "garáž"}

version_now = hist[hist["valid_to"].isna()]
view = in_window(version_now, WINDOW_MAP)
colors = [USAGE_COLORS.get(u, "#cfcec9") for u in view["usage_type_code"]]

fig, ax = plt.subplots(figsize=(6.3, 5.0))
draw_objects(ax, view, colors, WINDOW_MAP)
ax.set_title("Mapa objektů – barva podle způsobu využití")

handles = [plt.Line2D([0], [0], marker="s", linestyle="", color=c, markersize=8, label=USAGE_LABELS[k])
           for k, c in USAGE_COLORS.items()]
handles.append(plt.Line2D([0], [0], marker="s", linestyle="", color="#cfcec9", markersize=8, label="ostatní"))
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=4, fontsize=8, title="Barva podle: způsob využití",
          title_fontsize=8)

example = view[(view["usage_type_code"] == "6") & view["floor_count"].notna() & view["unit_count"].notna()]
example = example.sort_values("built_up_area", ascending=False).iloc[0]
ax.annotate(
    f"bytový dům\n{int(example['floor_count'])} podlaží, {int(example['unit_count'])} bytů\nzastavěno {example['built_up_area']:.0f} m²",
    (example["x"], example["y"]), xytext=(example["x"] + 130, example["y"] + 130), fontsize=7.5,
    bbox={"boxstyle": "round,pad=0.4", "fc": "white", "ec": INK_SOFT},
    arrowprops={"arrowstyle": "-", "color": INK_SOFT},
)
save(fig, "28_priklad_mapa_atribut")
plt.show()
len(view)

### Příklad 3 – časová osa a posuvník data

In [ ]:
WINDOW_TIME = (500, 1300, 2300, 3100)
DATES = [pd.Timestamp("2015-09-01"), pd.Timestamp("2021-01-01"), pd.Timestamp("2026-08-01")]


def valid_at(frame, t):
    return frame[(frame["valid_from"] <= t) & (frame["valid_to"].isna() | (frame["valid_to"] > t))]


area_hist = in_window(hist, WINDOW_TIME)
baseline_codes = set(valid_at(area_hist, DATES[0])["code"])

fig = plt.figure(figsize=(6.3, 3.6))
grid = fig.add_gridspec(2, 3, height_ratios=[4, 0.7], hspace=0.05, wspace=0.08)
counts = []
for i, t in enumerate(DATES):
    ax = fig.add_subplot(grid[0, i])
    at_t = valid_at(area_hist, t)
    colors = [INK_SOFT if c in baseline_codes else ORANGE for c in at_t["code"]]
    draw_objects(ax, at_t, ["#b9b8b3" if c in baseline_codes else ORANGE for c in at_t["code"]], WINDOW_TIME)
    ax.set_title(f"{t:%m/%Y}: {len(at_t)} objektů", fontsize=8, pad=3)
    counts.append(len(at_t))

slider = fig.add_subplot(grid[1, :])
span = (DATES[-1] - DATES[0]).days
for t in DATES:
    slider.plot([(t - DATES[0]).days / span] * 2, [0.3, 0.7], color=INK_SOFT, linewidth=1)
    slider.text((t - DATES[0]).days / span, 0.0, f"{t:%m/%Y}", ha="center", va="top", fontsize=7.5, color=INK_SOFT)
slider.plot([0, 1], [0.5, 0.5], color=GRID_COLOR if "GRID_COLOR" in globals() else "#cfcec9", linewidth=4, solid_capstyle="round")
slider.plot([0, 1], [0.5, 0.5], color=BLUE, linewidth=1.2)
slider.scatter([1.0], [0.5], s=70, color=BLUE, zorder=3)
slider.set_xlim(-0.05, 1.05)
slider.set_ylim(-0.6, 1)
slider.axis("off")
fig.suptitle("Stav území k vybranému datu (oranžově: přibylo po 09/2015)", fontsize=9.5,
             fontweight="bold", x=0.125, ha="left", y=0.99)
save(fig, "29_priklad_casova_osa")
plt.show()
counts

### Příklad 4 – časová řada změn

In [ ]:
changes_series = events["Změny atributů"].replace(0, np.nan)
mass_months = changes_series.nlargest(3).index

fig, ax = plt.subplots(figsize=(6.3, 3.2))
ax.bar(changes_series.index, changes_series.values, width=24,
       color=[ORANGE if m in mass_months else BLUE for m in changes_series.index])
ax.set_yscale("log")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: thousands(v)))
ax.axhline(changes_series.median(), color=INK_SOFT, linewidth=1, linestyle="--")
ax.text(changes_series.index[0], changes_series.median() * 0.78, f"medián {changes_series.median():.0f} změn za měsíc",
        fontsize=7.5, color=INK_SOFT, va="top", bbox={"fc": "white", "ec": "none", "pad": 1.5})
for m in mass_months:
    ax.annotate(f"{m:%m/%Y}: {thousands(changes_series[m])}", (m, changes_series[m]), xytext=(0, 4),
                textcoords="offset points", ha="center", fontsize=7.5, color=INK_SOFT)
ax.set_ylim(top=changes_series.max() * 2.5)
ax.set_ylabel("Změny atributů za měsíc (log. měřítko)")
ax.set_title("Časová řada změn s odlišenými hromadnými aktualizacemi")
ax.grid(axis="x", visible=False)
save(fig, "30_priklad_casova_rada")
plt.show()
mass_months

### Příklad 5 – detail objektu s historií verzí

In [ ]:
ATTRS = [("usage_type_code", "Způsob využití"), ("floor_count", "Počet podlaží"), ("unit_count", "Počet bytů"),
         ("built_up_area", "Zastavěná plocha (m²)")]
versions = hist.sort_values(["code", "valid_from"])
four = versions.groupby("code").filter(lambda g: len(g) == 4)
four = four.groupby("code").filter(lambda g: g[[a for a, _ in ATTRS]].notna().all().all())
changed = four.groupby("code")[[a for a, _ in ATTRS]].nunique().gt(1).sum(axis=1)
chosen = changed.sort_values(ascending=False).index[0]
obj = versions[versions["code"] == chosen].reset_index(drop=True)


def cell_text(attr, value):
    if attr == "usage_type_code":
        return usage_names.get(value, str(value))
    return f"{value:.0f}"


columns = []
for _, row in obj.iterrows():
    end = "dosud" if pd.isna(row["valid_to"]) else f"{row['valid_to']:%m/%Y}"
    columns.append(f"{row['valid_from']:%m/%Y} – {end}")
cells = [[cell_text(a, obj.loc[i, a]) for i in range(len(obj))] for a, _ in ATTRS]
fills = [[("#fbd9cb" if i > 0 and cells[r][i] != cells[r][i - 1] else "white") for i in range(len(obj))]
         for r in range(len(ATTRS))]

fig, ax = plt.subplots(figsize=(6.3, 2.6))
ax.axis("off")
table = ax.table(cellText=cells, rowLabels=[lbl for _, lbl in ATTRS], colLabels=columns, cellColours=fills,
                 colColours=["#e4e3df"] * len(obj), cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.6)
for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor("#cfcec9")
ax.set_title(f"Detail objektu {chosen}: verze a změněné údaje (oranžově)", fontsize=9.5, fontweight="bold",
             loc="left", y=0.97)
save(fig, "31_priklad_detail_objektu")
plt.show()
chosen, obj[["valid_from", "valid_to"] + [a for a, _ in ATTRS]]

### Příklad 6 – srovnání obcí

In [ ]:
largest8 = by_municipality.sort_values("objects", ascending=False).head(8).index
sewage_known = current[current["sewage_connection_code"].notna() & ~current["sewage_connection_code"].isin(["8", "9"])]
comparison = pd.DataFrame({
    "Rodinné\ndomy (%)": current.groupby("municipality_name")["usage_type_code"].apply(lambda s: (s == "7").mean() * 100),
    "Bytové\ndomy (%)": current.groupby("municipality_name")["usage_type_code"].apply(lambda s: (s == "6").mean() * 100),
    "Průměr\npodlaží": current.groupby("municipality_name")["floor_count"].apply(lambda s: s.dropna().astype(float).mean()),
    "Medián roku\ndokončení": year_geo.groupby("municipality_name")["year"].median(),
    "Kanalizace\n(%)": sewage_known.groupby("municipality_name")["sewage_connection_code"].apply(lambda s: (s == "1").mean() * 100),
    "Nová výstavba\n/ 1 000": by_municipality["per_1000"],
}).loc[largest8]

normed = (comparison - comparison.min()) / (comparison.max() - comparison.min())
fig, ax = plt.subplots(figsize=(6.3, 3.6))
ax.imshow(normed.values, cmap=SEQ, vmin=0, vmax=1.15, aspect="auto")
for i in range(comparison.shape[0]):
    for j in range(comparison.shape[1]):
        v = comparison.iloc[i, j]
        label = f"{v:.0f}" if comparison.columns[j].startswith(("Medián", "Nová")) else f"{v:.1f}" if "podlaží" in comparison.columns[j] else f"{v:.0f}"
        ax.text(j, i, label, ha="center", va="center", fontsize=8, color="white" if normed.iloc[i, j] > 0.5 else "#0b0b0b")
ax.set_xticks(range(comparison.shape[1]), comparison.columns, fontsize=7.5)
ax.xaxis.tick_top()
ax.set_yticks(range(comparison.shape[0]), [f"{n} ({thousands(by_municipality.loc[n, 'objects'])})" for n in comparison.index], fontsize=8)
ax.tick_params(length=0)
ax.grid(False)
for side in ax.spines.values():
    side.set_visible(False)
fig.suptitle("Srovnání obcí (barva = relativní hodnota ve sloupci, v závorce počet objektů)", fontsize=9.5, fontweight="bold",
             x=0.02, ha="left", y=0.98)
fig.tight_layout()
save(fig, "32_priklad_srovnani_obci")
plt.show()
comparison.round(1)

### Příklad 7 – rozdělení hodnot s filtrem

In [ ]:
FILTER_RANGE = (80, 150)
subset = current[current["is_brno"] & (current["usage_type_code"] == "7") & (current["built_up_area"] > 0)]
values = subset["built_up_area"]
bins = np.logspace(np.log10(20), np.log10(1000), 37)
counts, edges = np.histogram(values.clip(20, 1000), bins=bins)
inside = (edges[:-1] >= FILTER_RANGE[0] * 0.98) & (edges[1:] <= FILTER_RANGE[1] * 1.02)
n_inside = int(values.between(*FILTER_RANGE).sum())

fig = plt.figure(figsize=(6.3, 3.2))
grid = fig.add_gridspec(1, 2, width_ratios=[1.1, 2.2], wspace=0.3)
panel = fig.add_subplot(grid[0, 0])
panel.axis("off")
panel.text(0, 1.0, "Filtry", fontsize=9.5, fontweight="bold", va="top")
for k, (name, value) in enumerate([("Obec", "Brno"), ("Způsob využití", "rodinný dům"),
                                    ("Zastavěná plocha", f"{FILTER_RANGE[0]}–{FILTER_RANGE[1]} m²")]):
    y = 0.84 - k * 0.2
    panel.text(0, y, name, fontsize=7.5, color=INK_SOFT, va="top")
    panel.text(0.02, y - 0.07, value, fontsize=8.5, va="top",
               bbox={"boxstyle": "round,pad=0.3", "fc": "#cde2fb", "ec": BLUE})
panel.text(0, 0.1, f"{thousands(n_inside)} z {thousands(len(values))}\nobjektů vyhovuje", fontsize=8.5, va="top",
           fontweight="bold")

ax = fig.add_subplot(grid[0, 1])
ax.bar(edges[:-1], counts, width=np.diff(edges), align="edge", color=np.where(inside, BLUE, "#cfcec9"))
ax.set_xscale("log")
ax.set_xticks([20, 50, 100, 200, 500, 1000], ["20", "50", "100", "200", "500", "1\u00a0000"])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(thousands))
ax.set_xlabel("Zastavěná plocha (m², logaritmické měřítko)")
ax.set_ylabel("Počet objektů")
ax.set_title("Rozdělení zastavěné plochy s vybraným rozsahem")
ax.grid(axis="x", visible=False)
save(fig, "33_priklad_filtr")
plt.show()
n_inside, len(values)

### Příklad 8 – indikátor úplnosti

In [ ]:
COMPLETENESS = [("floor_count", "Počet podlaží"), ("built_up_area", "Zastavěná plocha"), ("completion_date", "Datum dokončení")]
by_name = current.groupby("municipality_name")
fill = pd.DataFrame({label: by_name[col].apply(lambda s: s.notna().mean() * 100) for col, label in COMPLETENESS}).loc[LABELLED]
fill = fill.loc[LABELLED[::-1]]

fig, axes = plt.subplots(1, 3, figsize=(6.3, 3.0), sharey=True)
for ax, label in zip(axes, fill.columns):
    ax.barh(fill.index, fill[label], color=BLUE, height=0.65)
    ax.barh(fill.index, 100 - fill[label], left=fill[label], color="#e4e3df", height=0.65)
    for y, v in enumerate(fill[label]):
        ax.text(2, y, f"{v:.0f} %", va="center", fontsize=7.5, color="white")
    ax.set_xlim(0, 100)
    ax.set_title(label, fontsize=8.5)
    ax.tick_params(labelsize=7.5)
    ax.grid(False)
fig.supxlabel("Podíl objektů s vyplněným údajem (modře) a bez údaje (šedě)", fontsize=8, color=INK_SOFT)
fig.suptitle("Indikátor úplnosti údajů podle obcí", x=0.02, ha="left", fontsize=9.5, fontweight="bold")
fig.tight_layout()
save(fig, "34_priklad_uplnost")
plt.show()
fill.round(0)